# PAEC Inference Pipeline

Run PAEC 2-pass inference with perspective prefixes and DS fusion.
Demonstrates both prompt-only (Phase 1) and prefix-based (Phase 2) modes.

In [ ]:
!pip install -q torch transformers peft datasets bitsandbytes accelerate tqdm pyyaml scipy scikit-learn
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/vualidon/tom_agent_collab.git /content/tom_agent_collab 2>/dev/null || (cd /content/tom_agent_collab && git pull)
%cd /content/tom_agent_collab

In [ ]:
import torch
from src.config import PAECConfig
from src.models.model_loader import load_base_model, load_model_with_prefix
from src.inference.perspective_runner import PerspectiveRunner

config = PAECConfig.from_yaml('configs/default.yaml')
runner = PerspectiveRunner(config)

# Test story: classic Sally-Anne false belief
STORY = 'Alice put the ball in the basket. Alice left the room. Bob moved the ball to the box.'
QUESTION = 'Where will Alice look for the ball?'
OPTIONS = ['basket', 'box']

In [ ]:
# Phase 1: Prompt-only PAEC (no prefix training needed)
base_model, tokenizer = load_base_model(config)

result_prompt = runner.predict(
    model_self=base_model, model_partner=base_model,
    tokenizer=tokenizer,
    story=STORY, question=QUESTION, options=OPTIONS,
    use_prompt_perspective=True,
)

print('=== Phase 1: Prompt-only PAEC ===')
print(f'Answer: {result_prompt.answer_text} ({OPTIONS[result_prompt.answer_idx]})')
print(f'Vacuity: {result_prompt.fused.vacuity:.4f}')
print(f'Dissonance: {result_prompt.fused.normalized_conflict:.4f}')
print(f'Confidence: {result_prompt.fused.confidence:.4f}')
print(f'Self belief: {result_prompt.opinion_self.belief}')
print(f'Partner belief: {result_prompt.opinion_partner.belief}')
print(f'Fused prob: {result_prompt.fused.projected_prob}')

In [ ]:
# Phase 2: Prefix-based PAEC
import gc
del base_model; gc.collect(); torch.cuda.empty_cache()

model_self, tokenizer = load_model_with_prefix(config, '/content/drive/MyDrive/paec_checkpoints/prefix_self')
model_partner, _ = load_model_with_prefix(config, '/content/drive/MyDrive/paec_checkpoints/prefix_partner')

result_prefix = runner.predict(
    model_self=model_self, model_partner=model_partner,
    tokenizer=tokenizer,
    story=STORY, question=QUESTION, options=OPTIONS,
)

print('=== Phase 2: Prefix-based PAEC ===')
print(f'Answer: {result_prefix.answer_text} ({OPTIONS[result_prefix.answer_idx]})')
print(f'Vacuity: {result_prefix.fused.vacuity:.4f}')
print(f'Dissonance: {result_prefix.fused.normalized_conflict:.4f}')
print(f'Confidence: {result_prefix.fused.confidence:.4f}')

In [ ]:
# Compare baselines
from src.baselines.standard_prompting import predict_standard
from src.baselines.simtom_prompting import predict_simtom
from src.baselines.self_consistency import predict_self_consistency

del model_self, model_partner; gc.collect(); torch.cuda.empty_cache()
base_model, tokenizer = load_base_model(config)

idx_std, conf_std = predict_standard(base_model, tokenizer, STORY, QUESTION, OPTIONS)
idx_sim, conf_sim = predict_simtom(base_model, tokenizer, STORY, QUESTION, OPTIONS, agent='Alice')
idx_sc2, conf_sc2 = predict_self_consistency(base_model, tokenizer, STORY, QUESTION, OPTIONS, n_samples=2)
idx_sc8, conf_sc8 = predict_self_consistency(base_model, tokenizer, STORY, QUESTION, OPTIONS, n_samples=8)

print(f'Standard:  {OPTIONS[idx_std]} (conf={conf_std:.3f})')
print(f'SimToM:    {OPTIONS[idx_sim]} (conf={conf_sim:.3f})')
print(f'SC-2:      {OPTIONS[idx_sc2]} (conf={conf_sc2:.3f})')
print(f'SC-8:      {OPTIONS[idx_sc8]} (conf={conf_sc8:.3f})')
print(f'PAEC(P):   {OPTIONS[result_prompt.answer_idx]} (conf={result_prompt.fused.confidence:.3f})')
print(f'PAEC(pfx): {OPTIONS[result_prefix.answer_idx]} (conf={result_prefix.fused.confidence:.3f})')